# Comparison of relative speeds of Pattern Matching Algorithms
## 1. Introduction and Problem Description

### Problem Statement
__Pattern matching__ is a fundamental problem in computer science that involves finding occurrences of a pattern string within a larger text string. This problem appears in numerous applications including:
- Text editors and word processors
- Search engines
- Bioinformatics (DNA sequence analysis)
- Network security (intrusion detection)
- Data mining and information retrieval

### Objective
In this project, we implement and compare three different pattern matching algorithms:
1. **Brute Force Algorithm**
2. **Boyer-Moore Algorithm** 
3. **Knuth-Morris-Pratt (KMP) Algorithm** 




We will analyze their performance characteristics, time complexity, and practical efficiency on various text datasets.

In [20]:
# Required for Python Packages
from random import randint, choice
import matplotlib.pyplot as plt


## **Algorithm Description**


### Brute Force Algorithm
The brute force algorithm is a straightforward approach to pattern matching. It checks for the pattern at every possible position in the text. If a mismatch occurs, it shifts the pattern by one position and continues checking.

In [11]:
def find_brute(T, P):
    n, m = len(T), len(P)
    if m > n:
        raise ValueError("Pattern length must be less than or equal to text length")
    for i in range(n - 1):
        
        k = 0
        while k < m and T[i + k] == P[k]:
            k += 1
            if k == m:
                return 'The pattern was found at index ' + str(i)
    return 'not found'

The worst-case time complexity of the first algorithm is $O(mn)$, where $m$ is the length of the pattern and $n$ is the length of the text. This is because in the worst case, the algorithm may need to check every character in the text for every character in the pattern.

### Boyer-Moore Algorithm
The __Boyer-Moore algorithm__ is a more efficient pattern matching algorithm then the brute force algorithm.
 
 It uses two heuristics to improve its runtime performance. It uses the __Looking-Glass__ heuristics, in which it compares the last index of the pattern, $P$, and compare with the text, $T$, at the current index. If it matches, it moves the backwards in the pattern index and compares the next left character. Another heuristics it uses is the __Character-Jump Heuristics__, which allows the algorithm to skip sections of the text that do not match the pattern. This is done by precomputing a table of character shifts based on the characters in the pattern.

In [12]:
def find_boyer_moore(T, P):
    """
    Boyer-Moore algorithm for string matching.
    """
    n, m = len(T), len(P) # length of text and pattern
    if m == 0: return 0 # search for empty string
    last = {} # build 'last' table
    for k in range(m):
        last[P[k]] = k # later occurrence overwrites 
    # align end of pattern at index m-1 of text
    i = m-1 # index of text
    k = m-1 # index of pattern
    while i < n:
        if T[i] == P[k]: # a matching character
            if k == 0:
                return 'The pattern was found at index ' + str(i)
            else:
                i -= 1 # examine previous character
                k -= 1 # of T and P
        else:
            j = last.get(T[i], -1) # last(T[i]) is -1 if not found
            i += m - min(k, j + 1) # case analysis for jump step
            k = m - 1 # reset at end of pattern
    return 'not found'


The Boyer-Moore algorithm has a worst-case time complexity of $O(mn)$, but in practice, it often performs much better than the brute force algorithm, especially for longer patterns and larger texts.

### Knuth-Morris-Pratt (KMP) Algorithm
The __Knuth-Morris-Pratt (KMP) algorithm__ is another efficient pattern matching algorithm that preprocesses the pattern to create a longest prefix-suffix (LPS) array. This array is used to skip unnecessary comparisons in the text when a mismatch occurs.
The KMP algorithm has a worst-case time complexity of $O(n + m)$, where $n$ is the length of the text and $m$ is the length of the pattern. This is because it processes each character in the text and pattern only once.



In [13]:
def compute_kmp_fail(P):
    """
    Utility that computes and returns KMP fail list.
        """
    m = len(P)
    fail = [0] * m # fail[i] is the length of the longest prefix of P[0:i+1]
    j = 0 # length of previous longest prefix
    for i in range(1, m):
        while j > 0 and P[i] != P[j]:
            j = fail[j - 1] # backtrack to previous longest prefix
        if P[i] == P[j]:
            j += 1
        fail[i] = j
    return fail

In [14]:
def find_kmp(T, P):
    """
    Knuth-Morris-Pratt algorithm for string matching.
    """
    # KMP search algorithm
    n, m = len(T), len(P)
    if m == 0: return 0 # search for empty string
    fail = compute_kmp_fail(P)
    j = 0 # index of text
    k = 0 # index of pattern
    while j < n:
        if T[j] == P[k]: # a matching character
            if k == m - 1: # end of pattern
                return 'The pattern was found at index ' + str(j - m + 1)
            j += 1 # examine next character
            k += 1 # of T and P
        elif k > 0:
            k = fail[k - 1]
        else:
            j += 1
    return 'not found'


The __KMP__ algorithm performs pattern matching on a text string of length n and a pattern string of length m in $O(n + m)$ time. It does this by preprocessing the pattern to create a longest __prefix-suffix__ (LPS) array, which is used to skip unnecessary comparisons in the text when a mismatch occurs.

## **Test Data Description**

### 3. Description of Test Documents

For comprehensive testing, we use multiple types of text documents with different characteristics:

#### Synthetic Text Data
- **Random text**: Randomly generated strings with uniform character distribution
- **Repetitive text**: Text with high repetition to test worst-case scenarios
- **DNA sequences**: Simulated genetic sequences with 4-character alphabet


#### Test Characteristics
- **Text sizes**: Ranging from 1,000 to 100,000 characters
- **Pattern lengths**: From 3 to 50 characters
- **Alphabet sizes**: 4 (DNA), 26 (lowercase), 95 (printable ASCII)
- **Pattern frequency**: Common and rare patterns

In [ ]:
class StringGenerator:
    """
    A class to generate random strings of a given length and character set.
    """
    def generateRandomString(self, length, charset='ABCDEFGHIJKLMNOPQRSTUVWXYZ'):
        """
        Generates a random string of a given length using the specified character set.
        """
        return ''.join(choice(charset) for _ in range(length))
    
    def generateRepetitiveString(self, length, charset='A'):
        """
        Generates a repetitive string of a given length using the specified character set.
        """
        pattern = ''.join(choice(charset) for _ in range(length))
        return pattern * (length // len(pattern)) + pattern[:length % len(pattern)]
    
    def generateDNAString(self, length):
        """
        Generates a random DNA string of a given length.
        """
        return self.generateRandomString(length, 'ACGT')

In [ ]:
generator = StringGenerator()

# Getting the strings for each size
sizes = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]

random_string = []
repetitive_string = []
dna_string = []

pattern_random_string = []
pattern_repetitive_string = []
pattern_dna_string = []

for i in range(len(sizes)):
    random_string.append(generator.generateRandomString(sizes[i]))
    repetitive_string.append(generator.generateRepetitiveString(sizes[i]))
    dna_string.append(generator.generateDNAString(sizes[i]))

# printing out the texts
"""
for i in range(len(sizes)):
    print(random_string[i])
    print(repetitive_string[i])
    print(dna_string[i])
"""

for i in range(len(sizes)):
    random_index = randint(0, sizes[i])
    # taking the pattern from the string
    pattern_random_string.append(random_string[i])
    # taking the pattern from the repetitive string
    pattern_repetitive_string.append(repetitive_string[i])
    # taking the pattern from the DNA string
    pattern_dna_string.append(dna_string[i])


...


## **Experimental Results and Analysis**

In [19]:
%timeit find_brute(random_string, pattern_random_string)
%timeit find_brute(repetitive_string, pattern_repetitive_string)
%timeit find_brute(dna_string, pattern_dna_string)

94.2 μs ± 1.58 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
26.6 μs ± 363 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
94.4 μs ± 2.29 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


# **Conclusion**
In this project, we implemented and compared three pattern matching algorithms: Brute Force, Boyer-Moore, and Knuth-Morris-Pratt. The results showed that: